# Топ-3 тикера по объёму торговли (MOEX OrderLog)

Читаем архивы `OrderLog*.zip` из `datasets/moex/orderlog/` напрямую из zip (без распаковки ~8 ГБ txt).

**Метрика:** суммарный **оборот в рублях** по сделкам (`ACTION=2`: `VOLUME × TRADEPRICE`).

Файлы обрабатываются **параллельно** (`ProcessPoolExecutor`): каждый zip — отдельный процесс.

Настройки во 2-й ячейке: `SELECTED_FILES`, `TOP_N`, `MAX_WORKERS`.

In [1]:
from __future__ import annotations

import os
import sys
import time
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "datasets").exists() and (ROOT.parent / "datasets").exists():
    ROOT = ROOT.parent

ORDERLOG_DIR = ROOT / "datasets" / "moex" / "orderlog"

# None = все zip в папке; или, например: ["OrderLog20250128.zip"]
SELECTED_FILES: list[str] | None = None

TOP_N = 3
# 0 = авто (min(cpu_count, число файлов))
MAX_WORKERS = 0

In [2]:
import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.orderlog_aggregate import aggregate_trades


def list_zip_files(orderlog_dir: Path, selected: list[str] | None) -> list[Path]:
    if selected:
        return [orderlog_dir / name for name in selected]
    return sorted(orderlog_dir.glob("OrderLog*.zip"))

In [3]:
zip_files = list_zip_files(ORDERLOG_DIR, SELECTED_FILES)
if not zip_files:
    raise FileNotFoundError(f"Нет OrderLog*.zip в {ORDERLOG_DIR}")

workers = MAX_WORKERS or min(os.cpu_count() or 4, len(zip_files))
print(f"Файлов к обработке: {len(zip_files)}, процессов: {workers}")
for p in zip_files:
    print(f"  {p.name} ({p.stat().st_size / 1e9:.2f} GB)")

total_turnover: dict[str, float] = defaultdict(float)
total_volume: dict[str, float] = defaultdict(float)
total_trades = 0
day_stats: list[tuple[str, int, list[tuple[str, float]]]] = []

t0 = time.perf_counter()
with ProcessPoolExecutor(max_workers=workers) as executor:
    futures = {executor.submit(aggregate_trades, str(p)): p for p in zip_files}
    done = 0
    for future in as_completed(futures):
        name, day_turnover, day_volume, day_trades = future.result()
        done += 1

        for ticker, value in day_turnover.items():
            total_turnover[ticker] += value
        for ticker, value in day_volume.items():
            total_volume[ticker] += value
        total_trades += day_trades

        top_day = sorted(day_turnover.items(), key=lambda x: x[1], reverse=True)[:3]
        day_stats.append((name, day_trades, top_day))
        top_str = ", ".join(f"{t} ({v / 1e9:.2f} млрд ₽)" for t, v in top_day)
        print(f"[{done}/{len(zip_files)}] {name}: {day_trades:,} сделок; топ: {top_str}")

elapsed = time.perf_counter() - t0
print(f"\nГотово за {elapsed / 60:.1f} мин")
print(f"Всего сделок: {total_trades:,}; тикеров: {len(total_turnover)}")

Файлов к обработке: 17, процессов: 16
  OrderLog20250103.zip (1.72 GB)
  OrderLog20250106.zip (1.25 GB)
  OrderLog20250108.zip (1.10 GB)
  OrderLog20250109.zip (2.09 GB)
  OrderLog20250110.zip (2.69 GB)
  OrderLog20250113.zip (2.64 GB)
  OrderLog20250114.zip (1.98 GB)
  OrderLog20250115.zip (1.79 GB)
  OrderLog20250116.zip (1.81 GB)
  OrderLog20250117.zip (1.60 GB)
  OrderLog20250120.zip (2.52 GB)
  OrderLog20250121.zip (1.82 GB)
  OrderLog20250122.zip (1.89 GB)
  OrderLog20250123.zip (1.61 GB)
  OrderLog20250124.zip (1.53 GB)
  OrderLog20250127.zip (1.64 GB)
  OrderLog20250128.zip (1.72 GB)


[1/17] OrderLog20250108.zip: 6,291,371 сделок; топ: T (26.90 млрд ₽), SBER (14.71 млрд ₽), LQDT (12.31 млрд ₽)


[2/17] OrderLog20250106.zip: 6,718,265 сделок; топ: T (29.71 млрд ₽), SBER (15.47 млрд ₽), LQDT (10.84 млрд ₽)


[3/17] OrderLog20250124.zip: 7,529,191 сделок; топ: SMLT (38.94 млрд ₽), LQDT (34.24 млрд ₽), PIKK (26.42 млрд ₽)


[4/17] OrderLog20250117.zip: 8,943,872 сделок; топ: SMLT (39.94 млрд ₽), LQDT (32.54 млрд ₽), T (31.25 млрд ₽)


...

[13/17] OrderLog20250109.zip: 7,647,608 сделок; топ: T (60.98 млрд ₽), X5 (37.21 млрд ₽), SBER (28.98 млрд ₽)


[14/17] OrderLog20250120.zip: 11,495,586 сделок; топ: SMLT (65.01 млрд ₽), T (41.54 млрд ₽), SBER (38.33 млрд ₽)


[15/17] OrderLog20250113.zip: 10,616,774 сделок; топ: T (98.62 млрд ₽), SBER (37.56 млрд ₽), LQDT (34.03 млрд ₽)


[16/17] OrderLog20250110.zip: 10,745,638 сделок; топ: T (96.09 млрд ₽), SBER (39.23 млрд ₽), LQDT (33.06 млрд ₽)


[17/17] OrderLog20250128.zip: 7,520,137 сделок; топ: LQDT (29.59 млрд ₽), T (28.29 млрд ₽), X5 (21.89 млрд ₽)

Готово за 4.5 мин
Всего сделок: 140,144,400; тикеров: 402


In [4]:
rows = []
for ticker, rub_turnover in total_turnover.items():
    rows.append(
        {
            "ticker": ticker,
            "turnover_rub": rub_turnover,
            "volume_shares": total_volume[ticker],
            "share_of_total_pct": 100 * rub_turnover / sum(total_turnover.values()),
        }
    )

df = pd.DataFrame(rows).sort_values("turnover_rub", ascending=False).reset_index(drop=True)
top = df.head(TOP_N).copy()
top["turnover_rub"] = top["turnover_rub"].map(lambda x: f"{x:,.0f}")
top["volume_shares"] = top["volume_shares"].map(lambda x: f"{x:,.0f}")
top["share_of_total_pct"] = top["share_of_total_pct"].map(lambda x: f"{x:.2f}%")

top.rename(
    columns={
        "ticker": "Тикер",
        "turnover_rub": "Оборот, ₽",
        "volume_shares": "Объём, шт.",
        "share_of_total_pct": "Доля оборота",
    },
    inplace=True,
)

print(f"Топ-{TOP_N} тикера по обороту торгов ({len(zip_files)} дн.)")
top

Топ-3 тикера по обороту торгов (17 дн.)


,Тикер,"Оборот, ₽","Объём, шт.",Доля оборота
0,T,"674,795,609,634","232,935,507",12.14%
1,LQDT,"461,873,370,671","292,575,298,259",8.31%
2,SBER,"442,564,559,060","1,586,334,390",7.96%
